# Week 3 Part 2 — Data Augmentation Local

Notebook này chỉ phục vụ phần tạo dữ liệu augment trên local.

Quy trình:
1. Setup môi trường local
2. Nạp API key
3. Sinh dữ liệu augment cho rare aspects
4. Lọc dữ liệu sinh ra
5. Kiểm tra thủ công chất lượng output
6. Lưu summary cho bước tiếp theo

Khuyến nghị:
- Chạy thử với `N_PER_ASPECT = 5` trước
- Khi ổn mới tăng lên `30`
- Chưa retrain trong notebook này

In [ ]:
# Cell 1 — Setup path  (LUON CHAY DAU TIEN sau moi kernel restart)
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/macbookpro/Documents/Master-study/NLP/absa-vlsp2018-hotel')
os.chdir(PROJECT_ROOT)

for p in ['code/week1', 'code/week3', 'code/week3_part2']:
    full = str(PROJECT_ROOT / p)
    if full not in sys.path:
        sys.path.insert(0, full)

print('cwd =', os.getcwd())
print('sys.path OK:', [p for p in sys.path if 'absa' in p])

In [2]:
# Cell 2 — Imports
import json
from collections import Counter

import pandas as pd

from utils.helpers import set_seed
from utils.constants import RARE_ASPECTS
from llm_client import LLMClient
from augmentor import run_augmentation_all_aspects
from augment_filter import filter_augmented_reviews

set_seed(42)
print('Imports OK')

Imports OK


In [ ]:
# Cell 3 — Cấu hình local run
# API key: set env var trước khi chạy notebook
#   export OPENAI_API_KEY="sk-..."
# KHÔNG hardcode key ở đây (tránh commit lên git)

PROVIDER = 'openai'
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

N_PER_ASPECT = 30
VERIFY_RATE = 0.3

api_key = OPENAI_API_KEY

if not api_key:
    raise ValueError(
        'Chua co OPENAI_API_KEY.\n'
        'Chay trong terminal: export OPENAI_API_KEY="sk-..."'
        ' roi mo lai notebook.'
    )

print('PROVIDER =', PROVIDER)
print('N_PER_ASPECT =', N_PER_ASPECT)
print('VERIFY_RATE =', VERIFY_RATE)
print('API key loaded =', bool(api_key))

In [4]:
# Cell 4 — Load train data và kiểm tra rare aspects
train_df = pd.read_csv('data/train_preprocessed.csv')
print('train size =', len(train_df))

for aspect in RARE_ASPECTS:
    count = int((train_df[aspect] > 0).sum())
    print(f'- {aspect}: {count}')

train size = 3000
- FACILITIES#MISCELLANEOUS: 33
- ROOM_AMENITIES#PRICES: 0
- ROOM_AMENITIES#MISCELLANEOUS: 3
- ROOM_AMENITIES#CLEANLINESS: 89
- ROOM_AMENITIES#DESIGN&FEATURES: 354
- HOTEL#DESIGN&FEATURES: 877
- ROOMS#MISCELLANEOUS: 5
- FOOD&DRINKS#MISCELLANEOUS: 13


In [5]:
# Cell 5 — Tạo LLM client
client = LLMClient(provider=PROVIDER, api_key=api_key)
print('LLM client ready')

LLM client ready


In [6]:
# Cell 6 — Generate augmented reviews
augmented = run_augmentation_all_aspects(
    train_df=train_df,
    llm_client=client,
    n_per_aspect=N_PER_ASPECT,
    output_path='data/augmented_reviews.json',
)

print('Generated total =', len(augmented))


Generating 30 reviews for: FACILITIES#MISCELLANEOUS
  [FACILITIES#MISCELLANEOUS] Batch 1/6: got 5 reviews (total: 5)
  [FACILITIES#MISCELLANEOUS] Batch 2/6: got 5 reviews (total: 10)


KeyboardInterrupt: 

In [ ]:
# Cell 7 — Kiểm tra raw output
with open('data/augmented_reviews.json', 'r', encoding='utf-8') as f:
    raw_augmented = json.load(f)

print('Raw generated count =', len(raw_augmented))
print(Counter(x.get('source_aspect', 'unknown') for x in raw_augmented))
raw_augmented[:3]

In [ ]:
# Cell 8 — Filter output
filtered = filter_augmented_reviews(
    raw_path='data/augmented_reviews.json',
    filtered_path='data/augmented_reviews_filtered.json',
    train_df=train_df,
    llm_client=client if VERIFY_RATE > 0 else None,
    verify_rate=VERIFY_RATE,
)

print('Filtered total =', len(filtered))

In [ ]:
# Cell 9 — Review thủ công dữ liệu sau filter
with open('data/augmented_reviews_filtered.json', 'r', encoding='utf-8') as f:
    filtered_data = json.load(f)

print('Filtered count =', len(filtered_data))
print(Counter(x.get('source_aspect', 'unknown') for x in filtered_data))

for idx, item in enumerate(filtered_data[:10], start=1):
    print('=' * 100)
    print('Sample', idx)
    print('source_aspect =', item.get('source_aspect'))
    print('review =', item.get('review'))
    print('labels =', item.get('labels'))

In [ ]:
# Cell 10 — Lưu summary nhanh cho Week 3 Part 2
summary = {
    'provider': PROVIDER,
    'n_per_aspect': N_PER_ASPECT,
    'verify_rate': VERIFY_RATE,
    'raw_count': len(raw_augmented),
    'filtered_count': len(filtered_data),
    'per_aspect_filtered': dict(Counter(x.get('source_aspect', 'unknown') for x in filtered_data)),
}

os.makedirs('outputs/results', exist_ok=True)
with open('outputs/results/week3_part2_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary

Sau khi xong, các file bạn sẽ dùng tiếp là:

- `data/augmented_reviews.json`
- `data/augmented_reviews_filtered.json`
- `outputs/results/week3_part2_summary.json`

Khi bạn xác nhận chất lượng tốt, bước sau mới làm merge + retrain.

## Phân tích lỗi & Fix (chạy lại từ đây)

**Vấn đề:** Lần chạy đầu cho kết quả 240 generated → 40 filtered (chỉ 5/aspect).

**Root cause — 2 bugs trong `augmentor.py`:**

1. `get_real_examples(random_state=42)` **cố định** ngoài vòng lặp → mọi batch dùng **cùng 3 examples** → LLM sinh **cùng 5 reviews** lặp 6 lần
2. `llm_client.complete(use_cache=True)` mặc định → cùng prompt = **cùng cache key** → response batch 1 được trả về cho tất cả batch sau

**Fix đã áp dụng trong `code/week3_part2/augmentor.py`:**
- `get_real_examples` gọi **bên trong** batch loop với `random_state=batch_idx*17+3` (khác mỗi batch)
- `llm_client.complete(use_cache=False)` để tránh cache
- Thêm danh sách reviews đã sinh vào prompt để LLM tránh lặp lại

**Kỳ vọng sau fix:** ~25–30 reviews/aspect qua được dedup (thay vì 5).

In [7]:
# Cell 11 — Reload module với fix mới
import importlib
import augmentor as aug_module
importlib.reload(aug_module)

from augmentor import run_augmentation_all_aspects

print("augmentor module reloaded (fixed version)")

augmentor module reloaded (fixed version)


In [8]:
# Cell 12 — Re-generate với code đã fix (ghi đè data/augmented_reviews.json)
augmented_v2 = run_augmentation_all_aspects(
    train_df=train_df,
    llm_client=client,
    n_per_aspect=N_PER_ASPECT,
    output_path='data/augmented_reviews.json',
)

print('Generated total =', len(augmented_v2))

# Kiểm tra nhanh unique count
from collections import Counter
counts = Counter(x['source_aspect'] for x in augmented_v2)
for asp, cnt in sorted(counts.items()):
    print(f'  {asp}: {cnt}')


Generating 30 reviews for: FACILITIES#MISCELLANEOUS
  [FACILITIES#MISCELLANEOUS] Batch 1/6: got 5 reviews (total: 5)
  [FACILITIES#MISCELLANEOUS] Batch 2/6: got 5 reviews (total: 10)
  [FACILITIES#MISCELLANEOUS] Batch 3/6: got 5 reviews (total: 15)
  [FACILITIES#MISCELLANEOUS] Batch 4/6: got 5 reviews (total: 20)
  [FACILITIES#MISCELLANEOUS] Batch 5/6: got 5 reviews (total: 25)
  [FACILITIES#MISCELLANEOUS] Batch 6/6: got 5 reviews (total: 30)
  -> Got 30 valid reviews

Generating 30 reviews for: ROOM_AMENITIES#PRICES
  [ROOM_AMENITIES#PRICES] Batch 1/6: got 5 reviews (total: 5)
  [ROOM_AMENITIES#PRICES] Batch 2/6: got 5 reviews (total: 10)
  [ROOM_AMENITIES#PRICES] Batch 3/6: got 5 reviews (total: 15)
  [ROOM_AMENITIES#PRICES] Batch 4/6: got 5 reviews (total: 20)
  [ROOM_AMENITIES#PRICES] Batch 5/6: got 5 reviews (total: 25)
  [ROOM_AMENITIES#PRICES] Batch 6/6: got 5 reviews (total: 30)
  -> Got 30 valid reviews

Generating 30 reviews for: ROOM_AMENITIES#MISCELLANEOUS
  [ROOM_AMENITIE

In [9]:
# Cell 13 — Kiểm tra diversity trước khi filter
import json

with open('data/augmented_reviews.json', 'r', encoding='utf-8') as f:
    raw_v2 = json.load(f)

# Đếm unique reviews (exact match)
unique_texts = set(r['review'] for r in raw_v2)
print(f'Total: {len(raw_v2)} reviews, Unique: {len(unique_texts)} ({len(unique_texts)/len(raw_v2)*100:.0f}%)')

# Per-aspect unique count
def char_ngrams(text, n=3):
    t = text.lower().strip()
    return frozenset(t[i:i+n] for i in range(len(t)-n+1))

from collections import defaultdict
by_aspect = defaultdict(list)
for r in raw_v2:
    by_aspect[r['source_aspect']].append(r)

print('\nPer-aspect unique check (dedup threshold=0.95):')
for asp in sorted(by_aspect.keys()):
    revs = by_aspect[asp]
    passed, existing = [], []
    for r in revs:
        ng = char_ngrams(r['review'])
        is_dup = any(len(ng & e) / len(ng | e) > 0.95 for e in existing if ng and e)
        if not is_dup:
            passed.append(r)
            existing.append(ng)
    print(f'  {asp}: {len(revs)} → {len(passed)} after dedup')

Total: 240 reviews, Unique: 232 (97%)

Per-aspect unique check (dedup threshold=0.95):
  FACILITIES#MISCELLANEOUS: 30 → 30 after dedup
  FOOD&DRINKS#MISCELLANEOUS: 30 → 30 after dedup
  HOTEL#DESIGN&FEATURES: 30 → 30 after dedup
  ROOMS#MISCELLANEOUS: 30 → 30 after dedup
  ROOM_AMENITIES#CLEANLINESS: 30 → 26 after dedup
  ROOM_AMENITIES#DESIGN&FEATURES: 30 → 30 after dedup
  ROOM_AMENITIES#MISCELLANEOUS: 30 → 28 after dedup
  ROOM_AMENITIES#PRICES: 30 → 28 after dedup


In [13]:
# Cell 14 — Re-filter (ghi đè data/augmented_reviews_filtered.json)
import importlib
import augment_filter as af_module
importlib.reload(af_module)
from augment_filter import filter_augmented_reviews

filtered_v2 = filter_augmented_reviews(
    raw_path='data/augmented_reviews.json',
    filtered_path='data/augmented_reviews_filtered.json',
    train_df=train_df,
    llm_client=client if VERIFY_RATE > 0 else None,
    verify_rate=VERIFY_RATE,
)

print('\nFiltered total =', len(filtered_v2))
print(Counter(x.get('source_aspect', 'unknown') for x in filtered_v2))


Filtering 240 generated reviews
[Heuristic filter] 240 -> 240 (removed: {'too_short': 0, 'too_long': 0, 'duplicate': 0, 'no_vietnamese': 0})
[LLM verify filter] Verified 72, removed 1 -> 239 reviews
[Dedup filter] 239 -> 231 (removed 8 duplicates)

Filtered: 231 reviews -> data/augmented_reviews_filtered.json

Filtered total = 231
Counter({'FACILITIES#MISCELLANEOUS': 30, 'ROOM_AMENITIES#DESIGN&FEATURES': 30, 'HOTEL#DESIGN&FEATURES': 30, 'ROOMS#MISCELLANEOUS': 30, 'FOOD&DRINKS#MISCELLANEOUS': 30, 'ROOM_AMENITIES#MISCELLANEOUS': 28, 'ROOM_AMENITIES#PRICES': 27, 'ROOM_AMENITIES#CLEANLINESS': 26})


In [14]:
# Cell 15 — Merge original train + augmented → train_augmented.csv
import pandas as pd
import json

orig_df = pd.read_csv('data/train_preprocessed.csv')
print(f'Original train: {len(orig_df)} samples')

with open('data/augmented_reviews_filtered.json', 'r', encoding='utf-8') as f:
    filtered_data = json.load(f)

from utils.constants import ASPECT_COLUMNS, LABEL_TO_IDX

rows = []
for item in filtered_data:
    row = {'Review': item['review'], 'processed_review': item['review']}
    for asp in ASPECT_COLUMNS:
        sentiment = item['labels'].get(asp, 'absent')
        row[asp] = LABEL_TO_IDX.get(sentiment, 0)
    rows.append(row)

aug_df = pd.DataFrame(rows)

# Đảm bảo cùng cột với orig_df
missing_cols = [c for c in orig_df.columns if c not in aug_df.columns]
for c in missing_cols:
    aug_df[c] = orig_df[c].iloc[0] if len(orig_df) > 0 else ''

aug_df = aug_df[orig_df.columns]  # sắp xếp cột giống nhau

merged = pd.concat([orig_df, aug_df], ignore_index=True)
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)
merged.to_csv('data/train_augmented.csv', index=False)

print(f'Augmented samples: {len(aug_df)}')
print(f'Merged train: {len(merged)} samples → data/train_augmented.csv')
print(f'Tăng {len(aug_df)/len(orig_df)*100:.1f}%')

Original train: 3000 samples
Augmented samples: 231
Merged train: 3231 samples → data/train_augmented.csv
Tăng 7.7%


In [15]:
# Cell 16 — Kiểm tra rare aspect counts trước/sau augmentation
import pandas as pd
from utils.constants import RARE_ASPECTS

orig_df = pd.read_csv('data/train_preprocessed.csv')
merged = pd.read_csv('data/train_augmented.csv')

print(f"{'Aspect':<40} {'Before':>8} {'After':>8} {'Delta':>8}")
print('-' * 68)
for asp in RARE_ASPECTS:
    before = int((orig_df[asp] > 0).sum())
    after = int((merged[asp] > 0).sum())
    print(f'{asp:<40} {before:>8} {after:>8} {after-before:>+8}')

print(f"\nTotal train: {len(orig_df)} → {len(merged)} (+{len(merged)-len(orig_df)})")

Aspect                                     Before    After    Delta
--------------------------------------------------------------------
FACILITIES#MISCELLANEOUS                       33       63      +30
ROOM_AMENITIES#PRICES                           0       27      +27
ROOM_AMENITIES#MISCELLANEOUS                    3       31      +28
ROOM_AMENITIES#CLEANLINESS                     89      115      +26
ROOM_AMENITIES#DESIGN&FEATURES                354      384      +30
HOTEL#DESIGN&FEATURES                         877      907      +30
ROOMS#MISCELLANEOUS                             5       35      +30
FOOD&DRINKS#MISCELLANEOUS                      13       43      +30

Total train: 3000 → 3231 (+231)
